This section gets the training code

In [ ]:
#Gets Nano GTP and training repositories
!git clone https://github.com/karpathy/nanogpt


#Gets code that I am using for training
!mkdir -p nanogpt/data/trainingcode

%cd nanogpt/data/trainingcode


#MIT Licenses
!git clone https://github.com/TheAlgorithms/Python.git
!git clone https://github.com/3b1b/manim.git
!git clone https://github.com/talkpython/python-for-absolute-beginners-course.git
!git clone https://github.com/pythonlessons/Reinforcement_Learning.git
!git clone https://github.com/mastering-python/code.git
!git clone https://github.com/rhettinger/modernpython
!git clone https://github.com/oanda/v20-python-samples.git
!git clone https://github.com/psf/black.git
!git clone https://github.com/pydantic/pydantic.git
!git clone https://github.com/Textualize/rich.git
!git clone https://github.com/fastapi/typer.git
!git clone https://github.com/polyglot-learn/LearnPython.git

Cloning into 'nanogpt'...
remote: Enumerating objects: 689, done.
remote: Total 689 (delta 0), reused 0 (delta 0), pack-reused 689 (from 1)
Receiving objects: 100% (689/689), 981.26 KiB | 15.58 MiB/s, done.
Resolving deltas: 100% (380/380), done.
/content/nanogpt/data/trainingcode
Cloning into 'Python'...
remote: Enumerating objects: 25134, done.
remote: Counting objects: 100% (14/14), done.
remote: Compressing objects: 100% (12/12), done.
remote: Total 25134 (delta 5), reused 2 (delta 2), pack-reused 25120 (from 2)
Receiving objects: 100% (25134/25134), 17.86 MiB | 30.48 MiB/s, done.
Resolving deltas: 100% (15573/15573), done.
Cloning into 'manim'...
remote: Enumerating objects: 42285, done.
remote: Total 42285 (delta 0), reused 0 (delta 0), pack-reused 42285 (from 1)
Receiving objects: 100% (42285/42285), 75.67 MiB | 44.25 MiB/s, done.
Resolving deltas: 100% (30836/30836), done.
Cloning into 'python-for-absolute-beginners-course'...
remote: Enumerating objects: 999, done.
remote: Cou

Installs certain things needed to look through Python directories

In [ ]:
!pip install transformers datasets tiktoken wandb tqdm


This goes through the path and gets only Python files

In [ ]:
%cd /content/nanogpt/data/trainingcode

from pathlib import Path
import hashlib
code_location = Path("/content/nanogpt/data/trainingcode")

accepted_files = []
seen = set()

counter = 0

# Goes through each python file and gathers it
for file in code_location.rglob("*.py"):
  try:
    #Gets the code
    code = file.read_text(encoding="utf-8")


    #This will make sure the code was not seen before
    seen_hash = hashlib.sha256(
        code.encode("utf-8")
    ).hexdigest()


    if seen_hash not in seen:
      seen.add(seen_hash)
      accepted_files.append(code)
    else:
      continue


  except Exception as e:
    continue


print(len(accepted_files))


/content/nanogpt/data/trainingcode
3526


This code below creates the training and validation set

In [ ]:
import random

random.seed(42)
random.shuffle(accepted_files)



#This will get the index, the files before will be with the original code
split_location = int(len(accepted_files) * 0.9)
train_files = accepted_files[:split_location]
test_files = accepted_files[split_location:]



print(f"There are {len(train_files)} training files and {len(test_files)} testing files")

There are 3173 training files and 353 testing files


This will split it up by function instead of file (Used AI)

In [ ]:
import ast
import hashlib

def extract_functions(items):
    functions = []
    seen_hashes = set()

    def visit(node, lines, path):
        # Extract functions, but don't recurse into them
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):

            if not hasattr(node, "end_lineno"):
                return

            name = node.name.lower()

            if name.startswith(("test_", "benchmark_")):
                return

            start_line = node.lineno

            # Include decorators
            if node.decorator_list:
                start_line = min(
                    start_line,
                    *(d.lineno for d in node.decorator_list)
                )

            function_text = "\n".join(
                lines[start_line - 1:node.end_lineno]
            ).strip()

            if not 80 <= len(function_text) <= 4000:
                return

            # Remove exact duplicates
            digest = hashlib.sha256(
                function_text.encode("utf-8")
            ).hexdigest()

            if digest in seen_hashes:
                return

            seen_hashes.add(digest)

            functions.append({
                "path": path,
                "text": function_text
            })

            return

        # Continue through modules and classes
        for child in ast.iter_child_nodes(node):
            visit(child, lines, path)

    # items is a list of plain strings (file contents)
    for text in items:
        try:
            tree = ast.parse(text)
        except (SyntaxError, ValueError):
            continue

        visit(tree, text.splitlines(), "")

    return functions


training_data = extract_functions(train_files)
test_data = extract_functions(test_files)

print("Training functions:", len(training_data))
print("Validation functions:", len(test_data))



<unknown>:15: SyntaxWarning: invalid escape sequence '\u'
<unknown>:16: SyntaxWarning: invalid escape sequence '\N'
<unknown>:36: SyntaxWarning: invalid escape sequence '\u'
<unknown>:37: SyntaxWarning: invalid escape sequence '\N'
<unknown>:169: SyntaxWarning: invalid escape sequence '\ '
<unknown>:175: SyntaxWarning: invalid escape sequence '\ '
<unknown>:180: SyntaxWarning: invalid escape sequence '\ '
<unknown>:185: SyntaxWarning: invalid escape sequence '\ '
<unknown>:232: SyntaxWarning: invalid escape sequence '\ '
<unknown>:403: SyntaxWarning: invalid escape sequence '\ '
<unknown>:409: SyntaxWarning: invalid escape sequence '\ '
<unknown>:414: SyntaxWarning: invalid escape sequence '\ '
<unknown>:420: SyntaxWarning: invalid escape sequence '\ '
<unknown>:467: SyntaxWarning: invalid escape sequence '\ '
<unknown>:108: SyntaxWarning: invalid escape sequence '\ '
<unknown>:114: SyntaxWarning: invalid escape sequence '\ '
<unknown>:119: SyntaxWarning: invalid escape sequence '\ '
<

Training functions: 11290
Validation functions: 1299
{'path': '', 'text': 'def word_occurrence(sentence: str) -> defaultdict[str, int]:\n    """\n    >>> from collections import Counter\n    >>> SENTENCE = "a b A b c b d b d e f e g e h e i e j e 0"\n    >>> occurrence_dict = word_occurrence(SENTENCE)\n    >>> all(occurrence_dict[word] == count for word, count\n    ...     in Counter(SENTENCE.split()).items())\n    True\n    >>> dict(word_occurrence("Two  spaces"))\n    {\'Two\': 1, \'spaces\': 1}\n    """\n    occurrence: defaultdict[str, int] = defaultdict(int)\n    # Creating a dictionary containing count of each word\n    for word in sentence.split():\n        occurrence[word] += 1\n    return occurrence'}


This does a similar thing with files where it makes them not duplicated

In [ ]:
def remove_duplicate_functions(funcs):
  seen_functions = set()

  returned_functions = []
  for func in funcs:
    if func["text"] in seen_functions:
      continue
    else:
      seen_functions.add(func["text"])
      returned_functions.append(func)

  return returned_functions



cleaned_training = remove_duplicate_functions(training_data)
cleaned_test = remove_duplicate_functions(test_data)

print("Training functions:", len(cleaned_training))
print("Validation functions:", len(cleaned_test))





Training functions: 11193
Validation functions: 1371


Attempts to prevent more than 1 function through tokens (Partially AI)

In [ ]:
# Define the end-of-text boundary token
END_OF_TEXT_TOKEN = "<|endoftext|>"

# Join the functions with the boundary token
train_corpus = f"\n{END_OF_TEXT_TOKEN}\n".join([func["text"] for func in cleaned_training]) # 'test' holds unique train functions
val_corpus = f"\n{END_OF_TEXT_TOKEN}\n".join([func["text"] for func in cleaned_test])

# Save the corpuses to disk
with open("train_corpus.txt", "w", encoding="utf-8") as f:
    f.write(train_corpus)

with open("val_corpus.txt", "w", encoding="utf-8") as f:
    f.write(val_corpus)


Creates a tokenizer (Partially AI)

In [ ]:
!pip install tokenizers

from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import ByteLevel
from tokenizers.decoders import ByteLevel as ByteLevelDecoder

# Train tokenizer ONLY on training split
with open("/content/training_corpus.txt", "w", encoding="utf-8") as f:
    f.write(train_corpus) # Use train_corpus which is the formatted string

tokenizer = Tokenizer(
    BPE(unk_token="<unk>")
)

tokenizer.pre_tokenizer = ByteLevel()

# ADD THIS
tokenizer.decoder = ByteLevelDecoder()

trainer = BpeTrainer(
    vocab_size=8192,
    special_tokens=[
        "<unk>",
        "<|endoftext|>"
    ]
)

tokenizer.train(
    files=["/content/training_corpus.txt"],
    trainer=trainer
)

# Encode the actual corpus strings created previously
training_tokens = tokenizer.encode(
    train_corpus
).ids

test_tokens = tokenizer.encode(
    val_corpus
).ids

print("Vocabulary size:", tokenizer.get_vocab_size())
print("Training tokens:", len(training_tokens))
print("Validation tokens:", len(test_tokens))

Vocabulary size: 8192
Training tokens: 2219226
Validation tokens: 259881


Tokenizer Part 2, idea was mine, partially helped code with AI

In [ ]:
import os
import pickle
import numpy as np

# Create the dataset directory
dataset_path = "/content/nanogpt/data/code_dataset"
os.makedirs(dataset_path, exist_ok=True)

# Save the tokens to binary files using uint16 (efficient up to vocab size 65,535)
np.array(training_tokens, dtype=np.uint16).tofile(os.path.join(dataset_path, "train.bin"))
np.array(test_tokens, dtype=np.uint16).tofile(os.path.join(dataset_path, "val.bin"))

# Save the tokenizer setup
tokenizer.save(os.path.join(dataset_path, "tokenizer.json"))

# Save metadata for nanoGPT to read
meta = {"vocab_size": tokenizer.get_vocab_size()}
with open(os.path.join(dataset_path, "meta.pkl"), "wb") as f:
    pickle.dump(meta, f)

print("Saved train.bin, val.bin, tokenizer.json, and meta.pkl successfully!")



train_data = np.fromfile(
    "/content/nanogpt/data/code_dataset/train.bin",
    dtype=np.uint16
)

val_data = np.fromfile(
    "/content/nanogpt/data/code_dataset/val.bin",
    dtype=np.uint16
)


print(
    "Train tokens:",
    len(train_data)
)

print(
    "Validation tokens:",
    len(val_data)
)

Saved train.bin, val.bin, tokenizer.json, and meta.pkl successfully!
Train tokens: 2219226
Validation tokens: 259881


Runs model, ideas were mine, some help coding new subjects through AI

In [ ]:
%cd /content/nanogpt

import csv
import os
import re
import subprocess

loss_csv_path = "/content/nanogpt/out-code-long/lossesonehead.csv"
os.makedirs(os.path.dirname(loss_csv_path), exist_ok=True)

command = [
    "python", "-u", "train.py",
    "--dataset=code_dataset",
    "--out_dir=out-code-long",
    '--init_from=scratch',
    "--block_size=512",
    "--batch_size=3",
    "--n_layer=8",
    "--n_head=8",
    "--n_embd=512",
    "--max_iters=3000",
    "--eval_interval=100",
    "--eval_iters=200",
    "--learning_rate=3e-4",
    "--warmup_iters=100",
    "--lr_decay_iters=3000",
    "--min_lr=3e-5",
    "--always_save_checkpoint=True",
    "--dropout=0.2",
    "--dtype=float16",
    "--device=cuda",
    "--compile=False",
]

loss_pattern = re.compile(
    r"step\s+(\d+):\s+train loss\s+([0-9.eE+-]+),\s+val loss\s+([0-9.eE+-]+)"
)

with open(loss_csv_path, "w", newline="") as csv_file:
    writer = csv.writer(csv_file)
    writer.writerow(["iteration", "train_loss", "val_loss"])
    csv_file.flush()

    process = subprocess.Popen(
        command,
        cwd="/content/nanogpt",
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    for line in process.stdout:
        print(line, end="")

        match = loss_pattern.search(line)
        if match:
            iteration = int(match.group(1))
            if iteration % 100 == 0:
                writer.writerow([
                    iteration,
                    float(match.group(2)),
                    float(match.group(3)),
                ])
                csv_file.flush()

    return_code = process.wait()

if return_code != 0:
    print(f"Command failed with return code {return_code}")
    raise subprocess.CalledProcessError(return_code, command)

from google.colab import files

checkpoint_path = "/content/nanogpt/out-code-long/ckpt.pt"

if not os.path.exists(checkpoint_path):
    raise FileNotFoundError(f"Checkpoint not found: {checkpoint_path}")

files.download(checkpoint_path)
files.download(loss_csv_path)
files.download("/content/nanogpt/data/code_dataset/tokenizer.json")

/content/nanogpt
Overriding: dataset = code_dataset
Overriding: out_dir = out-code-long
Overriding: init_from = scratch
Overriding: block_size = 512
Overriding: batch_size = 3
Overriding: n_layer = 8
Overriding: n_head = 8
Overriding: n_embd = 512
Overriding: max_iters = 3000
Overriding: eval_interval = 100
Overriding: eval_iters = 200
Overriding: learning_rate = 0.0003
Overriding: warmup_iters = 100
Overriding: lr_decay_iters = 3000
Overriding: min_lr = 3e-05
Overriding: always_save_checkpoint = True
Overriding: dropout = 0.2
Overriding: dtype = float16
Overriding: device = cuda
Overriding: compile = False
tokens per iteration will be: 61,440
found vocab_size = 8192 (inside data/code_dataset/meta.pkl)
Initializing a new model from scratch
number of parameters: 29.37M
/content/nanogpt/train.py:196: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(dtype == 'float16

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>



```
# This will run the model (usually used different file but can be done ere)
```



In [ ]:
import torch
from tokenizers import Tokenizer
from model import GPTConfig, GPT

device = "cuda"

# Loads tokenizer and runs model
tokenizer = Tokenizer.from_file(
    "/content/nanogpt/data/code_dataset/tokenizer.json"
)

# Load model
checkpoint = torch.load(
    "/content/nanogpt/out-code-long/ckpt.pt",
    map_location=device
)

config = GPTConfig(
    **checkpoint["model_args"]
)

model = GPT(config)

state_dict = checkpoint["model"]

# Handle nanoGPT checkpoint prefixes
for key in list(state_dict.keys()):
    if key.startswith("_orig_mod."):
        state_dict[key[len("_orig_mod."):]] = state_dict.pop(key)

model.load_state_dict(state_dict)

model.to(device)
model.eval()


number of parameters: 5.25M


GPT(
  (transformer): ModuleDict(
    (wte): Embedding(8192, 256)
    (wpe): Embedding(512, 256)
    (drop): Dropout(p=0.2, inplace=False)
    (h): ModuleList(
      (0-3): 4 x Block(
        (ln_1): LayerNorm()
        (attn): CausalSelfAttention(
          (c_attn): Linear(in_features=256, out_features=768, bias=False)
          (c_proj): Linear(in_features=256, out_features=256, bias=False)
          (attn_dropout): Dropout(p=0.2, inplace=False)
          (resid_dropout): Dropout(p=0.2, inplace=False)
        )
        (ln_2): LayerNorm()
        (mlp): MLP(
          (c_fc): Linear(in_features=256, out_features=1024, bias=False)
          (gelu): GELU(approximate='none')
          (c_proj): Linear(in_features=1024, out_features=256, bias=False)
          (dropout): Dropout(p=0.2, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm()
  )
  (lm_head): Linear(in_features=256, out_features=8192, bias=False)
)

Runs

In [ ]:
prompts = [
    """def sum_list(items):
    total = 0
    for item in items:
""",

    """def is_even(n):

""",

    """def find_max(items):
    largest = items[0]
    for item in items[1:]:
""",

    """def contains_duplicate(items):
    seen = set()
    for item in items:
        if item in seen:
""",

    """def linear_search(items, target):
    for i, item in enumerate(items):
        if item == target:
"""
]

for prompt in prompts:
    ids = tokenizer.encode(prompt).ids

    x = torch.tensor(
        ids,
        dtype=torch.long,
        device=device
    )[None, :]

    with torch.no_grad():
        y = model.generate(
            x,
            max_new_tokens=25,
            temperature=0.6,
            top_k=3
        )

    output = tokenizer.decode(y[0].tolist())

    print("=" * 60)
    print(output)

 def sum_list(items):
    total = 0
    for item in items:
        if item == 0:
            return max(item)
    return total
 
def __init__(
 def is_even(n):
   
        return True
    return False
 
def get_even_prime_prime(n: int, n
 def find_max(items):
    largest = items[0]
    for item in items[1:]:
        return sorted(item)
    return sorted(items, item)
 
def find_search(items
 def contains_duplicate(items):
    seen = set()
    for item in items:
        if item in seen:
            if item is not in item:
                return False
        if item in item:
                return True
   
 def linear_search(items, target):
    for i, item in enumerate(items):
        if item == target:
            return False
    return True
 
def get_sort_sort_sort_sort_sort(values
